In [1]:
import torch
from torch import nn
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from model.self_attention import SingleHeadAttention

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, mask: bool = False):
                super().__init__()
                self.attention_heads = nn.ModuleList()
                for _ in range(num_heads):
                    self.attention_heads.append(SingleHeadAttention(model_dim, model_dim // num_heads, mask))
                self.compute = nn.Linear(model_dim, model_dim)
                self.dropout = nn.Dropout(0.2)

    def forward(self, query, key=None, value=None):
        """
        query: Tensor of shape (B, T_q, D)
        key:   Tensor of shape (B, T_k, D) or None (defaults to query)
        value: Tensor of shape (B, T_v, D) or None (defaults to key)
        """
        if key is None:
            key = query
        if value is None:
            value = key

        head_outputs = []
        for head in self.attention_heads:
            head_outputs.append(head(query, key, value))

        concatenated = torch.cat(head_outputs, dim = -1)
        return self.dropout(self.compute(concatenated))

In [4]:
embedding_dim = 3
model_dim = 3
num_heads = 1

multi_headed_attention = MultiHeadedAttention(model_dim, num_heads, mask=True)
multi_headed_attention = multi_headed_attention.to(device)

embedded = [
    [[-1.4381, 0.1232, 0.5],
     [-0.1080, 0.3458, -0.2]],
    [[0.1929, -0.8567, 1.1],
     [-0.1160, 1.2547, 0.3]]
]
embedded = torch.tensor(embedded, dtype=torch.float32).to(device)
output = multi_headed_attention(embedded)
print(output)

tensor([[[0.0000, 0.5941, 0.0000],
         [0.9639, 0.0000, 0.5262]],

        [[0.2703, 0.6441, 0.7679],
         [0.0000, 0.4506, 0.5346]]], device='cuda:0',
       grad_fn=<NativeDropoutBackward0>)
